# Task 1 hyperparameter sweeps

One cell per model. `run_grid_search(task, model, parameter_grid)`:

- **a list** in `parameter_grid` = values to try (every combination of all lists is run)
- **a single value** (or a one-element list) = constant in every run
- **not mentioned** = the value from `submission/lib/config.py` (Task 1 section)

Every run trains on the 90 % training split and is scored on the 10 % validation split (validation MSE).
No refit on 100 %, nothing is written to `trained_models/` -> the submitted models stay untouched.

**Result:** `output/task1/<model>_<start time>.json` with the best combination and its full parameter set
(`best.parameters`, can be copied into `lib/config.py` as it is), the swept and constant values and the score of every run.
The file is rewritten after every run: interrupting a cell keeps the finished runs.

**Optional:** `rerun_finalists_on_other_splits=True` re-trains the best `n_finalists` (default 3) combinations on the
validation splits with seeds 1 and 2; the winner is then the best mean over the three splits (2 extra runs per finalist).
`show_training_log=True` prints the epoch lines of every run.

Run time per run on the RTX 5080 (Task 1 sweep, 7 Nov): SVD (k = 120) ~80 s, SVD++ (k = 500) ~6 min, bias model a few seconds.

In [1]:
%load_ext autoreload
%autoreload 2

# grid_search.py lives in this folder and imports assignment1.py / lib/ from ../submission
import torch

from grid_search import print_sweep_overview, run_grid_search

print(torch.__version__, "| CUDA available:", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no GPU")

2.14.1+cu130 | CUDA available: True | NVIDIA GeForce RTX 5080


## Bias model (closed form)

In [9]:
# a few seconds per run
bias_model_sweep = run_grid_search("task1", "bias_model", {
    "user_bias_regularization": [0.5, 1.0, 2.0, 3.0],
    "item_bias_regularization": [3.0, 5.0, 8.0, 16.0, 24.0],
    "n_iterations": [10, 20, 30, 50, 100],
})

task1 / bias_model: 100 combinations | swept: {'user_bias_regularization': [0.5, 1.0, 2.0, 3.0], 'item_bias_regularization': [3.0, 5.0, 8.0, 16.0, 24.0], 'n_iterations': [10, 20, 30, 50, 100]} | constant: -
-> C:\Dateien\Workspace\UCSD\Recommender_Systems\recommender-systems\assignment1_2026\sweep\output\task1\bias_model_2026-11-08_06-58-26.json
[1/100] user_bias_regularization=0.5 item_bias_regularization=3 n_iterations=10 ...
    validation MSE 0.76321 | 0 s | best so far 0.76321
[2/100] user_bias_regularization=0.5 item_bias_regularization=3 n_iterations=20 ...
    validation MSE 0.76306 | 0 s | best so far 0.76306
[3/100] user_bias_regularization=0.5 item_bias_regularization=3 n_iterations=30 ...
    validation MSE 0.76303 | 0 s | best so far 0.76303
[4/100] user_bias_regularization=0.5 item_bias_regularization=3 n_iterations=50 ...
    validation MSE 0.76301 | 1 s | best so far 0.76301
[5/100] user_bias_regularization=0.5 item_bias_regularization=3 n_iterations=100 ...
    validat

## SVD

In [3]:
# ~80 s per run (k = 120)
svd_sweep = run_grid_search("task1", "svd", {
    "user_bias_regularization": [1.0, 3.0],
    "item_bias_regularization": [3.0, 5.0],
    "n_factors": 120,
})

task1 / svd: 4 combinations | swept: {'user_bias_regularization': [1.0, 3.0], 'item_bias_regularization': [3.0, 5.0]} | constant: {'n_factors': 120}
-> C:\Dateien\Workspace\UCSD\Recommender_Systems\recommender-systems\assignment1_2026\sweep\output\task1\svd_2026-11-08_06-37-31.json
[1/4] user_bias_regularization=1 item_bias_regularization=3 ...
    validation MSE 0.74895 (epoch 7) | 84 s | best so far 0.74896
[2/4] user_bias_regularization=1 item_bias_regularization=5 ...
    validation MSE 0.74856 (epoch 7) | 82 s | best so far 0.74856
[3/4] user_bias_regularization=3 item_bias_regularization=3 ...
    validation MSE 0.75021 (epoch 7) | 82 s | best so far 0.74856
[4/4] user_bias_regularization=3 item_bias_regularization=5 ...
    validation MSE 0.74975 (epoch 7) | 83 s | best so far 0.74856
best: user_bias_regularization=1 item_bias_regularization=5 -> validation MSE 0.74856


## SVD++

In [ ]:
# ~6 min per run (k = 500) -> 4 runs ~25 min; with the finalist check (2 finalists x 2 splits) ~50 min
svdpp_sweep = run_grid_search("task1", "svdpp", {
    "user_bias_regularization": [1.0, 3.0],
    "item_bias_regularization": [3.0, 5.0],
}, rerun_finalists_on_other_splits=False, n_finalists=2)

## Factorization machine

In [10]:
factorization_machine_sweep = run_grid_search("task1", "factorization_machine", {
    "n_factors": [20, 60, 100, 140],
    "n_activity_buckets": [5, 10, 20],
    "factor_regularization": [5.0, 10.0, 15.0, 20.0],
    "ratings_per_batch": [256, 512, 1024],
})

task1 / factorization_machine: 144 combinations | swept: {'n_factors': [20, 60, 100, 140], 'n_activity_buckets': [5, 10, 20], 'factor_regularization': [5.0, 10.0, 15.0, 20.0], 'ratings_per_batch': [256, 512, 1024]} | constant: -
-> C:\Dateien\Workspace\UCSD\Recommender_Systems\recommender-systems\assignment1_2026\sweep\output\task1\factorization_machine_2026-11-08_07-01-19.json
[1/144] n_factors=20 n_activity_buckets=5 factor_regularization=5 ratings_per_batch=256 ...


    validation MSE 0.75802 (epoch 5) | 120 s | best so far 0.75802
[2/144] n_factors=20 n_activity_buckets=5 factor_regularization=5 ratings_per_batch=512 ...
    validation MSE 0.75948 (epoch 5) | 59 s | best so far 0.75802
[3/144] n_factors=20 n_activity_buckets=5 factor_regularization=5 ratings_per_batch=1024 ...
    validation MSE 0.76039 (epoch 5) | 37 s | best so far 0.75802
[4/144] n_factors=20 n_activity_buckets=5 factor_regularization=10 ratings_per_batch=256 ...
    validation MSE 0.75761 (epoch 6) | 130 s | best so far 0.75761
[5/144] n_factors=20 n_activity_buckets=5 factor_regularization=10 ratings_per_batch=512 ...
    validation MSE 0.75919 (epoch 6) | 66 s | best so far 0.75761
[6/144] n_factors=20 n_activity_buckets=5 factor_regularization=10 ratings_per_batch=1024 ...
    validation MSE 0.76032 (epoch 6) | 40 s | best so far 0.75761
[7/144] n_factors=20 n_activity_buckets=5 factor_regularization=15 ratings_per_batch=256 ...
    validation MSE 0.75747 (epoch 9) | 168 s

KeyboardInterrupt: 

## Item-item kNN

In [ ]:
# CPU, a few minutes per run (neighbourhoods of all items)
item_knn_sweep = run_grid_search("task1", "item_knn", {
    "n_neighbors": [25, 50, 100],
    "similarity_shrinkage": [80.0, 100.0, 120.0],
    "max_ratings_per_user_for_similarity": [100, 200, 300, 400],
    "items_per_similarity_block": [1000, 2000, 3000],
})

task1 / item_knn: 3 combinations | swept: {'n_neighbors': [25, 50, 100]} | constant: {'similarity_shrinkage': 100.0}
-> C:\Dateien\Workspace\UCSD\Recommender_Systems\recommender-systems\assignment1_2026\sweep\output\task1\item_knn_2026-11-08_06-47-39.json
[1/3] n_neighbors=25 ...
    validation MSE 0.75658 | 19 s | best so far 0.75658
[2/3] n_neighbors=50 ...
    validation MSE 0.75636 | 19 s | best so far 0.75636
[3/3] n_neighbors=100 ...
    validation MSE 0.75615 | 19 s | best so far 0.75615
best: n_neighbors=100 -> validation MSE 0.75615


## Overview

In [6]:
# one line per sweep file in output/task1 (also works right after a kernel restart)
print_sweep_overview("task1")

bias_model_2026-11-08_06-36-37.json                finished         12 / 12 | best 0.76284 with user_bias_regularization=1 item_bias_regularization=5
factorization_machine_2026-11-08_06-43-13.json     finished           3 / 3 | best 0.75520 with factor_regularization=16
item_knn_2026-11-08_06-47-39.json                  finished           3 / 3 | best 0.75615 with n_neighbors=100
svd_2026-11-08_06-37-31.json                       finished           4 / 4 | best 0.74856 with user_bias_regularization=1 item_bias_regularization=5
